## What this notebook does

Builds and evaluates **X1**, the first of the two proposed methods (proposal deck, slide 4): a
cluster-based reciprocal recommender that never needs a training history for either
participant, which matters because leave-one-wave-out CV is a *total* cold start (notebook 01,
Section 15; notebook 02, Section 15).

1. **Choosing k for each gender** — silhouette and interpretability on the full-sample person
   table, before any fold is fitted.
2. **The compatibility tables** — men are clustered on `FEATURES_SELFRATE` (what they say they
   are like), women on `FEATURES_PREF` (what they say they want); the two cluster-label sets are
   cross-tabulated into directional yes-rate and match-rate tables, fit on training-wave rows
   only, inside each fold, via `x1_cluster.fit_x1`.
3. **Evaluation protocol** — the same metric implementations (`evaluation.py`) the baseline
   uses, so results are comparable rather than a second, incompatible scoring scheme.
4. **Head-to-head against baseline Z** — X1 vs. predicted desirability, both scored
   identically, over the 21 folds. Evaluation in this notebook is against the baseline alone —
   no naive-constant floor is computed.
5. **Discussion and hand-off** — what the comparison means, and open items for X2.

Every step below has a short explanation of *why* we do it and *what to notice* in its result,
the same convention notebook 01 uses.

> **Sources.** `x1_cluster.py` is the source of truth this notebook narrates, the same
> convention `sd_features.py` follows for notebook 02. The proposal deck
> (`reciprocal_rs_slide.pptx`) fixes X1 as one of the two proposed methods; `report.qmd`'s
> evaluation section is a rubric, not a fixed metric spec — `03_Baseline_Desirability_
> Model.ipynb` is what made the concrete choice (**Recall@3, NDCG@3, Coverage@3**) this notebook
> follows.

## What this notebook does *not* do

It does not re-derive the feature table (notebook 02 does that) or implement X2 or the final
cross-method comparison in the report. It also does not tune `k` by grid search against the
evaluation metric — `k` is chosen once, up front, by interpretability and silhouette, exactly
as a cold-start clustering step should be: the CV loop must not see the held-out wave at any
point, including through hyperparameter selection.

## Setup

Same libraries as the earlier notebooks, plus this notebook's own source-of-truth modules --
`x1_cluster` (the clustering and compatibility-table logic) and `evaluation` (the metrics shared
with the baseline) -- and `baseline_desirability`, needed only for the head-to-head comparison
in Section 4.

In [1]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from sd_features import (load_features, load_raw, build_person_table, FEATURES_PREF,
                         FEATURES_SELFRATE, SEED)
from x1_cluster import fit_x1, predict_x1, SELFRATE_COLS, PREF_CLUSTER_COLS, PREF_ZERO_FLAG_COLS
from evaluation import evaluate
from baseline_desirability import wilson_desirability, make_profile_preprocessor

feat = load_features()
print(f"{feat.shape[0]} directed encounters, {feat['iid'].nunique()} people, "
      f"{feat['wave'].nunique()} waves")

8368 directed encounters, 551 people, 21 waves


## 1. Choosing k for each gender

`fit_x1` needs one `k` per gender before it can cluster a single fold, and that `k` must not be
chosen using information from a held-out wave -- so we fix it here, once, on the full sample
(551 people, 277 men / 274 women), standardised and median-imputed, by silhouette and
interpretability. The real fits happen per-fold, on training waves only, inside `fit_x1`; this
scan is exploratory and never touches the CV loop.

Women's preferences are scanned on 4 of the 6 stated-preference shares --
`pref_attr, pref_sinc, pref_intel, pref_fun` -- not all 6, because `pref_amb` and `pref_shar` are
**zero for 12 and 14 of 274 women respectively** (signup allocated all 100 points elsewhere).
The six shares are compositional (they sum to ~1 per person), so those zeros would dominate
K-Means -- both on the raw shares and on a CLR (centered-log-ratio) transform, which turns a
near-zero share into a large-magnitude outlier in log-space. Silhouette on the CLR version
looked *better* (0.58 vs 0.34 at k=2) purely because it was isolating the ~14 zero-`pref_shar`
women as their own cluster, not because it found a richer preference typology. Both zero flags
are kept instead as separate per-person indicators (`pref_zero_flags` in `x1_cluster.py`),
available to whoever consumes X1's output, without letting them distort the cluster geometry.

In [2]:
person = feat.drop_duplicates("iid").set_index("iid")
men_all = person[person["gender"] == 1]
women_all = person[person["gender"] == 0]

def k_scan(df, cols, label, k_range=range(2, 7)):
    X = StandardScaler().fit_transform(SimpleImputer(strategy="median").fit_transform(df[cols]))
    print(f"\n{label} (n={len(df)})")
    print(f"{'k':>2}  {'inertia':>10}  {'silhouette':>10}")
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit(X)
        sil = silhouette_score(X, km.labels_)
        print(f"{k:>2}  {km.inertia_:>10.1f}  {sil:>10.3f}")

k_scan(men_all, SELFRATE_COLS, "MEN on FEATURES_SELFRATE")
k_scan(women_all, PREF_CLUSTER_COLS, "WOMEN on 4 non-zero-inflated FEATURES_PREF")


MEN on FEATURES_SELFRATE (n=277)
 k     inertia  silhouette
 2      1004.9       0.262
 3       880.3       0.172


 4       787.9       0.161


 5       724.0       0.162
 6       661.5       0.163

WOMEN on 4 non-zero-inflated FEATURES_PREF (n=274)
 k     inertia  silhouette
 2       864.5       0.454
 3       711.7       0.329


 4       584.2       0.327


 5       498.3       0.316
 6       456.0       0.266


**What to notice:** silhouette peaks at k=2 for both groups (a coarse "modest vs. confident"
split for men, a "balanced vs. looks-first" split for women) and is essentially flat from k=3
upward -- there is no sharp elbow recommending a specific larger k. **We use k=4 for both
genders**: it is the smallest k that separates out an interpretable, distinct minority type on
each side (a small attractiveness-maximising cluster among women; a small
below-average-on-everything cluster among men) while keeping every cluster large enough that
its compatibility-table cells still get a reasonable number of training rows (roughly 260-550
people / 4 clusters / 2 directions per cell, before shrinkage).

## 2. The compatibility tables, made concrete on one fold

Two cluster labels per person are not yet a recommender -- we still need a way to turn (his
cluster, her cluster) into a predicted compatibility score. The simplest option that keeps X1
fully interpretable is a lookup table: cross-tabulate the two cluster-label sets and read off
how often that cell said yes or matched in training, rather than fitting a separate classifier
whose coefficients would need their own explanation.

`fit_x1` clusters men and women separately, then cross-tabulates the two cluster-label sets
into three k_men x k_women tables: `compat_female_dec` (how often the woman in that cell said
yes), `compat_male_dec` (how often the man did), `compat_match` (how often they mutually
matched). Every cell is shrunk toward that fold's global rate with `m=20` pseudo-counts -- the
same convention `sd_features.add_behavioural_features` uses, and for the same reason: most
cells here have plenty of rows, but shrinkage costs nothing and protects the rare small one.

Fitting on wave 1 held out (i.e. training on waves 2-21) as a concrete example:

In [3]:
example_train = feat[feat["wave"] != 1]
example_model = fit_x1(example_train)

print("compat_match -- P(mutual match) by (male self-rating cluster, female preference cluster):")
print(example_model.compat_match.round(3))
print(f"\nGlobal match rate this fold: {example_model.global_match:.3f}")

compat_match -- P(mutual match) by (male self-rating cluster, female preference cluster):
female_cluster      0      1      2      3
male_cluster                              
0               0.183  0.181  0.183  0.149
1               0.202  0.150  0.194  0.164
2               0.099  0.140  0.100  0.133
3               0.164  0.143  0.137  0.143

Global match rate this fold: 0.161


**What to notice:** male cluster 2 (the small, below-average-on-everything self-rating group)
matches below this fold's global rate (0.161) against *every* female cluster -- 0.099 to 0.140,
never above it. No female cluster shows the same pattern in reverse: female cluster 1 (the
largest, most balanced preference group) gets a fairly similar match rate from every male
cluster (0.181, 0.150, 0.140, 0.143) -- a much narrower spread than male cluster 2's row. On
this one fold, the male self-rating axis is doing more of the discriminating than the female
preference axis, which is worth watching across folds rather than reading into a single
example.

## 3. Evaluation protocol

Before X1 can be compared to anything, it has to be scored the same way the baseline already
is -- otherwise an apparent difference between methods could just be a difference between
yardsticks. `evaluation.py` is the metric module shared with the baseline, so every method is
scored identically. It implements the concrete metric set the baseline already committed to --
**Recall@3, NDCG@3, Coverage@3** -- not the NDCG@3/Precision@3/MRR/Coverage@3 list an earlier
draft of this notebook used, which predated the baseline's implementation.

The ranking unit is one decider in one wave: every candidate they met that night, ordered by
the model's score for that row. **Relevance is `match`**, not `dec` -- the point of a
*reciprocal* recommender is ranking people who would actually match highly, not just people the
decider alone would say yes to. The score used for ranking is `x1_score`: the directional table
appropriate to the row's own decider (`x1_female_dec` on a woman's row, `x1_male_dec` on a
man's), not `x1_match` -- the same convention `report.qmd`'s method section describes and
notebook 05's comparison pipeline uses, so this notebook's numbers agree with the final report.

**Coverage@3** is defined exactly as the baseline defines it: the fraction of distinct
candidate *people* (`pid`) recommended in *someone's* top-3 that wave, out of all distinct
candidates available that wave -- a people-coverage number, not a row-coverage number.

**Ties are broken by a seeded shuffle, not row order.** The baseline's own signal diagnostic
(section 8 of that notebook) found predicted desirability is heavily compressed toward the
mean, so ties are common; X1's cluster lookup is more compressed still (at most k_men x
k_women = 16 distinct scores per fold). An unbroken sort would silently favour whichever row
happens to sit first in the raw data on every tie -- a bigger, more misleading effect for a
coarse method like X1 than for a continuous one, so `evaluation.py` shuffles each candidate
list before the stable sort, exactly like the baseline's `_rank_by_score`.

In [4]:
fold_metrics = []
all_scored = []

for w in sorted(feat["wave"].unique()):
    train = feat[feat["wave"] != w]
    test = feat[feat["wave"] == w].copy()

    model = fit_x1(train)
    preds = predict_x1(model, test)

    scored = pd.DataFrame({
        "decider_iid": test["iid"].to_numpy(),
        "candidate_id": test["pid"].to_numpy(),
        "score": preds["x1_score"].to_numpy(),
        "match": test["match"].to_numpy(dtype="float64"),
    }, index=test.index)
    all_scored.append(scored)

    m = evaluate(scored)
    m["wave"], m["n_rows"] = w, len(test)
    fold_metrics.append(m)

fold_df = pd.DataFrame(fold_metrics).set_index("wave")

print("Per-fold X1 metrics:")
print(fold_df.round(3))

Per-fold X1 metrics:
      Recall@3  NDCG@3  Precision@3    MRR  Coverage@3  n_rows
wave                                                          
1        0.487   0.474        0.426  0.562       0.750     200
2        0.279   0.254        0.213  0.363       0.743     608
3        0.344   0.280        0.200  0.432       0.750     200
4        0.206   0.302        0.292  0.502       0.611     648
5        0.372   0.475        0.444  0.667       0.789     180
6        0.562   0.446        0.250  0.525       0.900      50
7        0.139   0.207        0.179  0.429       0.812     512
8        0.269   0.257        0.235  0.399       0.700     200
9        0.206   0.253        0.245  0.422       0.625     800
10       0.221   0.236        0.238  0.401       0.778     162
11       0.073   0.133        0.111  0.345       0.405     882
12       0.127   0.139        0.111  0.297       0.464     392
13       0.211   0.195        0.156  0.356       0.579     180
14       0.080   0.154        0.14

In [5]:
cols = ["Recall@3", "NDCG@3", "Coverage@3"]
summary = pd.DataFrame({
    "mean (X1, 21 folds)": fold_df[cols].mean(),
    "std (X1, 21 folds)": fold_df[cols].std(),
})
summary

,"mean (X1, 21 folds)","std (X1, 21 folds)"
Recall@3,0.238650,0.136342
NDCG@3,0.253600,0.109712
Coverage@3,0.649053,0.138125


**What to notice:** X1's Recall@3 and NDCG@3 average 0.239 and 0.254 across the 21 folds, with a
fold-to-fold standard deviation of roughly 0.11-0.14 (see `summary` above) — on their own these
numbers say nothing about whether X1 has learned anything real, because there is no reference
point yet. This notebook does not score a naive constant ranking to supply that reference:
`01_Speed_Dating_EDA.ipynb` Section 15.1 already rejects a global-constant ranking as making
"the comparison meaningless". Evaluation here is against the baseline alone — a fitted
competitor, not a no-signal floor. That comparison is Section 4, against baseline Z.

## 4. Head-to-head against baseline Z (predicted desirability)

As Section 3 noted, this notebook does not score a naive constant ranking —
`01_Speed_Dating_EDA.ipynb` Section 15.1 already rejects that as making "the comparison
meaningless" — so the evaluation here is against the baseline alone: **Z**, exactly as
implemented in `baseline_desirability.py`: a `RandomForestRegressor` trained per fold on
training-wave participants' signup profiles to predict their Wilson-shrunk desirability, applied
to the held-out wave, ranking each decider's candidates by the *candidate's own* predicted
desirability (never the decider's — that is what keeps Z a non-personalised popularity ranking).
Refit here, per fold, using the shared `evaluate()` so the comparison is exact.

In [6]:
raw = load_raw()
raw_person = build_person_table(raw)

head_to_head = {"X1 (clusters)": [], "Baseline Z (predicted desirability)": []}

for w in sorted(feat["wave"].unique()):
    train = feat[feat["wave"] != w]
    test = feat[feat["wave"] == w].copy()

    base_cols = pd.DataFrame({
        "decider_iid": test["iid"].to_numpy(),
        "candidate_id": test["pid"].to_numpy(),
        "match": test["match"].to_numpy(dtype="float64"),
    }, index=test.index)

    # X1
    x1_model = fit_x1(train)
    x1_preds = predict_x1(x1_model, test)
    scored = base_cols.copy()
    scored["score"] = x1_preds["x1_score"].to_numpy()
    head_to_head["X1 (clusters)"].append(evaluate(scored))

    # Baseline Z
    train_iids = raw.loc[raw["wave"] != w, "iid"].unique()
    test_iids = raw.loc[raw["wave"] == w, "iid"].unique()
    y_train = wilson_desirability(raw, train_iids).dropna()
    X_train = raw_person.loc[raw_person.index.isin(y_train.index)]
    y_train = y_train.loc[X_train.index]

    prep = make_profile_preprocessor()
    rf = RandomForestRegressor(n_estimators=300, max_depth=6, min_samples_leaf=5, random_state=SEED)
    rf.fit(prep.fit_transform(X_train), y_train.to_numpy())

    X_test = raw_person.loc[raw_person.index.isin(test_iids)]
    pred_desirability = pd.Series(rf.predict(prep.transform(X_test)), index=X_test.index)

    scored = base_cols.copy()
    scored["score"] = scored["candidate_id"].map(pred_desirability)
    head_to_head["Baseline Z (predicted desirability)"].append(evaluate(scored))

h2h_mean = pd.DataFrame({name: pd.DataFrame(v).mean() for name, v in head_to_head.items()}).T
h2h_std = pd.DataFrame({name: pd.DataFrame(v).std() for name, v in head_to_head.items()}).T
print("Mean across 21 folds:")
display(h2h_mean.round(4))
print("\nStd dev across 21 folds:")
display(h2h_std.round(4))

Mean across 21 folds:


,Recall@3,NDCG@3,Precision@3,MRR,Coverage@3
X1 (clusters),0.2386,0.2536,0.2102,0.4173,0.6491
Baseline Z (predicted desirability),0.3502,0.3483,0.2604,0.4799,0.2759



Std dev across 21 folds:


,Recall@3,NDCG@3,Precision@3,MRR,Coverage@3
X1 (clusters),0.1363,0.1097,0.094,0.1014,0.1381
Baseline Z (predicted desirability),0.2164,0.1790,0.124,0.1428,0.1308


**What to notice:** X1 trails baseline Z on both accuracy metrics -- Recall@3 (0.239 vs 0.350)
and NDCG@3 (0.254 vs 0.348) -- but has more than double Z's Coverage@3 (0.649 vs 0.276). This is
not "X1 loses"; it is a real accuracy-vs-diversity tradeoff, and it is exactly the failure mode
`01_Speed_Dating_EDA.ipynb` Section 9 measured directly: "a popularity ranker recommends the
same small set of highly-desired people to *everybody*". Z's accuracy comes partly from doing
precisely that -- concentrating recommendations on whoever it predicts is broadly desirable,
which helps Recall/NDCG but caps how many distinct people ever get surfaced. X1's cluster-pair
lookup has no notion of "universally desirable" at all -- every prediction is specific to a
(his-type, her-type) cell, so different deciders with different types get routed to different
top-3s, which is why coverage is so much higher even though the cluster grid is coarse.

## 5. Discussion and hand-off

**What this sets up for X2:** the factorization machine's job is not just to beat Z on
accuracy -- a model that only concentrates harder on "safe" popular candidates would repeat Z's
coverage problem. The target implied by X1 and Z together is Z's Recall@3/NDCG@3 (or better)
*without* collapsing back to Z's ~0.28 coverage.

**Open items for X2, and the final comparison):**
1. `k=4` was chosen once, by interpretability, not tuned against these metrics -- re-running
   this notebook with a different k is cheap if X2's results suggest the grid is too coarse or
   too fine.
2. `x1_cluster.py` exports the raw cluster ids (`male_cluster`, `female_cluster`) and the two
   zero-preference flags alongside the compatibility scores in `predict_x1`'s output -- X2 can
   use the cluster ids as categorical features rather than only consuming X1's final score.
3. Report standard deviations alongside every mean here (`h2h_std` above) -- both this
   notebook's and the baseline's per-fold spread are large relative to the gaps between
   methods, so the final comparison should say whether X2's improvement is bigger than fold
   variance, not just report means.